# Audio Transcriber

Based on https://github.com/google-gemini/cookbook/blob/main/quickstarts/Audio.ipynb

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

### Install dependencies

In [ ]:
!pip install -q -U google-genai # gemini interaction
!pip install -q -U python-docx # reading sample word documents
!pip install -q -U pydub # audio processing

!apt-get update
!apt-get install -y ffmpeg # audio processing

In [ ]:
# general python libraries
import os
import math

# gemini specific libraries
from google import genai
from google.colab import drive
from google.colab import userdata

# word document processing
from docx import Document

# audio processing
from pydub import AudioSegment

### Load API Key

The API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`.
You can get an API key from https://aistudio.google.com/app/u/1/apikey.

In [ ]:
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

### Mount Google Drive

Your Google Drive will be mounted in `/content/drive`.

In [ ]:
drive.mount('/content/drive')

## Utility Functions

Define utility functions to, for example, handle audio files.

In [ ]:
def list_files(file_path):
  return [f.path for f in sorted(os.scandir(file_path), key=lambda x: (x.is_dir(), x.name)) if f.is_file()]

def split_audio_files(input_files, max_length, output_directory):
  print(f'splitting {len(input_files)} files into {max_length/1000/60}min slots')

  for input_file in input_files:
    file = AudioSegment.from_mp3(input_file)
    file_length = len(file)
    slots = file[::max_length]
    for idx, slot in enumerate(slots):
      slot.export(f'{output_directory}/{os.path.basename(input_file)}.{idx}.mp3', format='mp3')

## Prompts, Models and Configuration

Verify that the prompts are correct and set the Gemini model to be used for each part of the transcription.

Make sure to set `drive_path` correctly!

Pick a model from the [AI Studio](https://aistudio.google.com/u/1/prompts/new_chat).

In [ ]:
# defines the path to use for processing
drive_path = 'audio_transcription'
path = f'/content/drive/MyDrive/{drive_path}'
paths = {
  'audio_files': f'{path}/audio_files',
  'processing': f'{path}/processing',
  'example_summaries': f'{path}/example_summaries'
}

# defines the used prompts for each part of the processing
prompts = {
  'transcription': (
    'Listen carefully to the following audio files in German.'
    'You are a medical writer and are required to transcribe the audio.'
    'Use paragraphs to separate people speaking. Try to annotate each paragraph with the person who spoke.'
    'Make sure to match the speakers with their voices in the entire transcript.'
    'If you cannot match the voice to a name, make sure to group the spoken paragraphs by the corresponding voice and for each voice use a different placeholder.'
    'For example, you can use "Voice 1" for one voice and "Voice 2" for a different one.'
    'Listen to it twice to verify and get it correct. Names always take precendence.'
    'If you are not sure how to transcribe a part, place [...] in the text and do not assume anything.'
    'Match voices as good as possible to their names.'
  ),
  'agenda': (
    'You are a medical writer and are required to write a medical protocol of the meeting.'
    'Summarize:'
    '- the agenda'
    '- the attendees with a brief profile, grouped by the organization they work for'
    '- the discussed core topics'
    '- the next steps'
    '- any important emphasis or other relevant topics'
    'Make sure it is all related to the meeting notes and medical topics.'
  ),
  'summary': (
    'You are a medical writer and are required to write a medical protocol of the meeting.'
    'Include as many details in the protocol as possible. Try to not summarize too much but rather keep it more descriptive for the protocol.'
  )
}

# defines the models for each part of the processing
models = {
  'transcription': 'gemini-2.5-pro-preview-05-06', # gemini-1.5-pro
  'agenda': 'gemini-2.5-flash-preview-04-17', # gemini-2.0-flash
  'summary': 'gemini-2.5-pro-preview-05-06' # gemini-1.5-pro
}

# defines the output files
outputs = {
  'transcription': f'{path}/transcription.txt',
  'agenda': f'{path}/agenda.txt',
  'summary': f'{path}/summary.txt'
}

# defines audio details
max_audio_length = 20*60*1000 # 20 minutes in milliseconds

## Create Directories / Upload Files

The files to be transcribed must be uploaded in a directory named `audio_files`.

If you want to generate a summary and have example files, place them into the `example_summaries` directory.

In [ ]:
os.makedirs(paths['audio_files'], exist_ok=True)
os.makedirs(paths['example_summaries'], exist_ok=True)
os.makedirs(paths['processing'], exist_ok=True)

### **Now upload your files!**

## Preprocessing

In [ ]:
for filename in os.listdir(paths['processing']):
   file_path = os.path.join(paths['processing'], filename)
   if os.path.isfile(file_path):
      os.remove(file_path)

split_audio_files(list_files(paths['audio_files']), max_audio_length, paths['processing'])
audio_files = list_files(paths['processing'])
print(f'found files: {audio_files}')

## Transcription

In [ ]:
print('creating the transcription')

try:
  os.remove(outputs['transcription'])
except OSError:
  pass

transcription = ''

for file in audio_files:
  file_name = os.path.basename(file)
  print(f'transcribing {file_name}')
  uploaded_file = client.files.upload(file=file)

  with open(outputs['transcription'], 'a') as output:
    response = client.models.generate_content_stream(
      model=models['transcription'],
      contents=[
        prompts['transcription'],
        uploaded_file
      ]
    )
    output.write('---\n')
    output.write(f'file: {file_name}')
    output.write('\n\n')
    for part in response:
        output.write(part.text)
        transcription += part.text
    output.write('\n\n')

  transcription += '\n'

print(f'transcription file: {outputs["transcription"]}')

## Agenda

In [ ]:
print('creating the agenda from the transcription')

try:
  os.remove(outputs['agenda'])
except OSError:
  pass

response = client.models.generate_content_stream(
  model=models['agenda'],
  contents=[
      prompts['agenda'],
      transcription
  ]
)

with open(outputs['agenda'], 'w') as output:
  for part in response:
    output.write(part.text)

print(f'agenda file: {outputs["agenda"]}')

## Summary

In [ ]:
print('creating the summary from the transcription')

try:
  os.remove(outputs['summary'])
except OSError:
  pass

example_files = list_files(paths['example_summaries'])
print(f'found example files: {example_files}')

examples = []
for file in example_files:
  print(f'reading example: {os.path.basename(file)}')
  with open(file, 'rb') as doc:
    document = Document(doc)
    examples.append('\n'.join([txt.text for txt in document.paragraphs]))

model_context = [
  prompts['summary'],
  transcription
]
if len(example_files) > 0:
  model_context = [
    prompts['summary_examples'],
    transcription,
    '---Examples---\n',
    '\n\n'.join(examples)
  ]

response = client.models.generate_content_stream(
  model=models['summary'],
  contents=model_context
)

with open(outputs['summary'], 'w') as output:
  for part in response:
    output.write(part.text)

print(f'summary file: {outputs["summary"]}')